## PaperTres

In [ ]:
# -*- conding:utf-8 -*-

import sys
from pathlib import Path

import hyperspy.api as hs
import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from matplotlib import gridspec, ticker, transforms
from matplotlib.colorbar import Colorbar
from matplotlib.colors import LinearSegmentedColormap, ListedColormap, to_rgba

In [ ]:
# Ensure custom module Path is set before import
sys.path.append(r"D:\CHENG\OneDrive - UAB\ICMAB-Python\Figure")
from colors import tol_cmap, tol_cset  # type: ignore

# 画图的初始设置
plt.style.use(r"D:\CHENG\OneDrive - UAB\ICMAB-Python\Figure\liuchzzyy.mplstyle")
# print(plt.style.available)  # noqa: ERA001

# xarray setting
xr.set_options(
    cmap_sequential="viridis",
    cmap_divergent="viridis",
    display_width=150,
)  # viridis, gray

# 颜色设定
colors = tol_cset("vibrant")
if colors is not None:
    colors = list(colors)
else:
    # Fallback colors in case tol_cset returns None
    colors = ["#0077BB", "#33BBEE", "#009988", "#EE7733", "#CC3311", "#EE3377", "#BBBBBB"]
if r"sunset" not in plt.colormaps():
    cmap = tol_cmap("sunset")
    if isinstance(cmap, LinearSegmentedColormap):
        plt.colormaps.register(cmap)
if r"rainbow_PuRd" not in plt.colormaps():
    cmap = tol_cmap("rainbow_PuRd")
    if isinstance(cmap, LinearSegmentedColormap):
        plt.colormaps.register(cmap)  # 备用 plasma

# 输出的文件夹
path_out = Path(r"C:\Users\chengliu\Desktop\Figure")

# Set math font
mpl.rcParams["mathtext.fontset"] = "custom"
mpl.rcParams["mathtext.rm"] = "Arial"
mpl.rcParams["mathtext.it"] = "Arial:italic"
mpl.rcParams["mathtext.bf"] = "Arial:bold"
mpl.rcParams["mathtext.sf"] = "Arial"
mpl.rcParams["mathtext.tt"] = "Arial"
mpl.rcParams["mathtext.cal"] = "Arial"
mpl.rcParams["mathtext.default"] = "regular"

### Figure 2

#### Echem

In [ ]:
# 对应实验上的电化学时间
path_filelist = list(
    Path(
        r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\Results\XAS\Operando\αMnO2\MeshMapping\2023-CLAESS\Data\case1_1stCharge_1stDischarge\Echem"  # noqa: E501, RUF001
    ).glob(r"*down*.txt")
)
# 读取电化学数据
echem = []
for path_file in path_filelist:
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break  # 发现后立即退出循环，提高效率

    df = pd.read_csv(
        path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal="."
    ).dropna(axis=1, how="all")  # noqa: E501
    # # 转换数据格式
    df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]].apply(
        pd.to_numeric, errors="coerce"
    )  # noqa: E501
    df["time/s"] = df["time/s"].apply(pd.to_datetime, format="mixed", errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    echem.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echem)):
    echem[i] = echem[i][echem[i].iloc[:, 0].isin([0, 1])]

data = pd.concat([echem[1], echem[0]], ignore_index=True, axis=0)
data.drop(columns=["Unnamed: 3"], inplace=True, errors="ignore")  # noqa: ERA001
data['time'] = (data['time/s'] - data['time/s'].iloc[0]).dt.total_seconds()

In [ ]:
# 谱线上的时间
path_file = Path(
    r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\Results\XAS\Operando\αMnO2\MeshMapping\2023-CLAESS\Results\V8\case1_1stCharge_1stDischarge"  # noqa: E501, RUF001
)
# 读取谱线的时间和ID
time_spectrum = pd.read_csv(
    Path.joinpath(path_file, r"Time_Index_Spectrum.csv"),
    sep=",",
    index_col=None,
    header=0,
    comment="#",
    date_format="%m/%d/%y %H:%M:%S.%f",
    parse_dates=[4],
)
time_spectrum["Time"] = pd.to_datetime(time_spectrum["Time"])
time_spectrum["ScanID"] = pd.to_numeric(time_spectrum["ScanID"])
# time_spectrum.info()  # noqa: ERA001

# 匹配谱线和电化学上的时间
echem_time = data["time/s"].values
target_times = time_spectrum.loc[
    (time_spectrum["Element"] == "Mn") &
    (time_spectrum["Sample"] == "DownCell"),
    "Time"
].values

index_spectrum_Mn = [  # noqa: N816
    np.abs(echem_time - t).argmin()
    for t in target_times
]

target_times = time_spectrum.loc[
    (time_spectrum["Element"] == "Zn") &
    (time_spectrum["Sample"] == "DownCell"),
    "Time"
].values
index_spectrum_Zn = [  # noqa: N816
    np.abs(echem_time - t).argmin()
    for t in target_times
]

# 提取对应的电位和电流数据，并按电位排序
Mn_info = time_spectrum.loc[
    (time_spectrum["Element"] == "Mn") &
    (time_spectrum["Sample"] == "DownCell"),
    ["Element", "Sample", "Energy"]
].reset_index(drop=True)

Mn_Voltage = data.loc[index_spectrum_Mn, ["Ewe/V", "<I>/mA"]].reset_index(drop=False)

index_voltage_Mn = pd.concat([Mn_Voltage, Mn_info], axis=1).reset_index(drop=False, inplace=False).sort_values(by="level_0")  # noqa: E501, N816

Zn_info = time_spectrum.loc[
    (time_spectrum["Element"] == "Zn") &
    (time_spectrum["Sample"] == "DownCell"),
    ["Element", "Sample", "Energy"]
].reset_index(drop=True)

Zn_Voltage = data.loc[index_spectrum_Zn, ["Ewe/V", "<I>/mA"]].reset_index(drop=False)

index_voltage_Zn = pd.concat([Zn_Voltage, Zn_info], axis=1).reset_index(drop=False, inplace=False).sort_values(by="level_0")  # noqa: E501, N816

In [ ]:
%matplotlib inline

# 画图
fig = plt.figure(figsize=(7.0, 2.5))
gs = gridspec.GridSpec(1, 1, width_ratios=None, height_ratios=None, wspace=0, hspace=0, figure=fig)

subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1, 1))
ax.set_box_aspect(0.4)

ax.plot(data["time"]/3600, data["Ewe/V"], ls="-", lw=1.0, c=colors[0], label=r"Voltage", zorder=0) # type: ignore

# Mn
for i, j in enumerate(index_voltage_Mn['Energy'].unique()[:1]):
    selected_times = data["time"].loc[index_voltage_Mn.loc[index_voltage_Mn["Energy"] == j, "index"]]
    selected_voltages = data["Ewe/V"].loc[index_voltage_Mn["index"][index_voltage_Mn["Energy"] == j]]
    ax.scatter(selected_times/3600, selected_voltages, c=colors[i], edgecolors="face", alpha=1.0, zorder=1) # type: ignore

# 添加索引文本标注
# 只标记一次，选取 r'6533' 的点
Index_special = [0, 5, 9, 15, 19, 27, 34, 41]
energy = index_voltage_Mn['Energy']
for i in [energy.unique()[0],]:
    row_index = np.array([
        np.abs(index_voltage_Mn[energy == i].loc[:, "level_0"] - val).argmin()
        for val in Index_special
    ])
    row_index = index_voltage_Mn[energy == i].iloc[row_index, 1]

    for j, idx in enumerate(row_index):
        ax.scatter(
            data["time"].iloc[idx], data["Ewe/V"].iloc[idx], c=colors[0], edgecolors="face", marker="o", zorder=2
        )
        ax.text(
            data["time"].iloc[idx]/3600,
            data["Ewe/V"].iloc[idx] + 0.03,
            str(Index_special[j]),
            fontsize=10,
            verticalalignment="bottom",
            horizontalalignment="right",
        )

# # Zn
# for i, j in enumerate(index_voltage_Zn['Energy'].unique()):
#     selected_times = data["time"].loc[index_voltage_Zn.loc[index_voltage_Zn["Energy"] == j, "index"]]
#     selected_voltages = data["Ewe/V"].loc[index_voltage_Zn["index"][index_voltage_Zn["Energy"] == j]]
#     ax.scatter(selected_times, selected_voltages, c=colors[i], edgecolors="face", marker='*', alpha=1.0, zorder=1)


ax.set_ylabel(
    r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$",
    fontsize=11,
)
ax.set_ylim(0.85, 1.85)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0.05))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0.05))

# 确保时间刻度从数据最开始时间显示
ax.set_xlim(0, 20)
ax.set_xlabel(r"Duration Time (hour)", fontsize=11,)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=4, offset=0.0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=2, offset=0.0))

ax.tick_params(
    axis="both", which="both", direction="out", labelsize=9, left=True, labelleft=True, top=False, labeltop=False
)
ax.legend(loc="upper left", bbox_to_anchor=(0.5, 1), frameon=False, fontsize=11)

ax.text(
    0.2,
    0.95,
    r"$\text{1.642 mg cm}^{-2}$",
    transform=ax.transAxes,
    fontsize=12,
    color=colors[3], # type: ignore
    va="top",
    ha="right",
    fontfamily="Arial",
)

ax2 = ax.twinx()
ax2.set_position((0, 0, 1, 1))
ax2.set_box_aspect(0.4)

ax2.plot(data["time"]/3600, data["<I>/mA"], ls="--", lw=1.0, c=colors[3], label=r"Current", zorder=0) # type: ignore

ax2.set_ylabel(
    r"Current (mA)",
    fontsize=11,
)
ax2.set_ylim(-0.2, 0.2)
ax2.yaxis.set_major_locator(ticker.MultipleLocator(base=0.1, offset=0))
ax2.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.05, offset=0))
ax2.tick_params(axis="both", which="both", direction="out", labelsize=9, right=True, labelright=True)

ax2.legend(loc="upper left", bbox_to_anchor=(0.65, 1), frameon=False, fontsize=11) # type: ignore

plt.savefig(
    Path.joinpath(path_out, r"PaperTres_Figure_02_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperTres_Figure_02_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.gcf().set_facecolor("white")
plt.show()

### Figure 1

In [ ]:
# 对应实验上的电化学时间
path_filelist = list(
    Path(
        r"D:\CHENG\OneDrive - UAB\ICMAB-Data\Zn-Mn\Results\XAS\Operando\αMnO2\MeshMapping\2023-CLAESS\Data\case1_1stCharge_1stDischarge\Echem"  # noqa: E501, RUF001
    ).glob(r"*down*.txt")
)
# 读取电化学数据
echem = []
for path_file in path_filelist:
    with open(path_file, "r", encoding="latin_1") as file:
        for line in file:
            if line.startswith("Nb header lines"):
                line_skip = int(line.split(":")[1].strip())
                break  # 发现后立即退出循环，提高效率

    df = pd.read_csv(
        path_file, sep="\t", comment="#", skiprows=line_skip - 1, encoding="latin_1", index_col=None, decimal="."
    ).dropna(axis=1, how="all")  # noqa: E501
    # # 转换数据格式
    df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]] = df[["Ewe/V", "<I>/mA", "Capacity/mA.h"]].apply(
        pd.to_numeric, errors="coerce"
    )  # noqa: E501
    df["time/s"] = df["time/s"].apply(pd.to_datetime, format="mixed", errors="coerce")
    df["cycle number"] = df["cycle number"].astype(float).astype(np.int16)
    echem.append(df)

# 选取每个数据的第一到第二圈
for i in range(len(echem)):
    echem[i] = echem[i][echem[i].iloc[:, 0].isin([0, 1])]

In [ ]:
%matplotlib inline
plt.close("all")

# 画图
fig = plt.figure(figsize=(3.3, 2.5))
gs = gridspec.GridSpec(1, 1, width_ratios=None, height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1, 1))
ax.set_box_aspect(0.8)
labels = [
    [r"$\mathrm{1^{st}}$", None, None, None],
    [r"$\mathrm{2^{nd}}$", None, None, None],
    [r"$\mathrm{3^{rd}}$", None, None, None],
    [r"$\mathrm{4^{th}}$", None, None, None],
]
for i, data in enumerate(echem):
    for j, idx in enumerate(data.iloc[:, 0].unique()):
        temp = data[data.iloc[:, 0] == idx].reset_index(drop=True)
        # 找到电压最小值的索引
        idx_min = temp["Ewe/V"].idxmin()
        # 断开最小值前后的曲线
        ax.plot(
            temp.loc[:idx_min, "Capacity/mA.h"] * 1000 / 0.821,
            temp.loc[:idx_min, "Ewe/V"],
            ls="-",
            lw=1.0,
            c=colors[j], # type: ignore
            label=labels[j][i],
            zorder=0,
        )
        ax.plot(
            temp.loc[idx_min+5:temp.shape[0]-2, "Capacity/mA.h"] * 1000 / 0.821,
            temp.loc[idx_min+5:temp.shape[0]-2, "Ewe/V"],
            ls="-",
            lw=1.0,
            c=colors[j], # type: ignore
            label=None,
            zorder=0,
        )

# 设置刻度线等格式
ax.set_xlabel(r"$\mathrm{Capacity \ (mAh \,g^{-1}_{MnO2})}$", fontsize=11, labelpad=1.0)
ax.set_xlim(0, 400)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=80, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=40, offset=0))

ax.set_ylabel(r"Voltage (V vs. Zn/Zn$\mathrm{^{2\!+}\!)}$", fontsize=11, labelpad=1.0)
ax.set_ylim(0.8, 2.0)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=0.2, offset=0.0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=0.1, offset=0.0))

ax.tick_params(axis="both", which='both', direction="out", labelsize=9)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 0.6), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)
ax.text(
    0.05,
    0.07,
    r"$\mathrm{0.5M \ ZnSO_4 + 0.2M \ MnSO_4}$",
    ha="left",
    va="top",
    transform=ax.transAxes,
    fontsize=9,
    c="k",
)
ax.text(
    0.05,
    0.20,
    r"$\mathrm{1.642 \ mg \ cm^{-2}}$",
    ha="left",
    va="top",
    transform=ax.transAxes,
    fontsize=9,
    c="k",
)

# 保存图片
plt.savefig(
    Path.joinpath(path_out, r"PaperTres_Figure_01_V0_300.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperTres_Figure_01_V0_600.tif"),
    pad_inches=0.05,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.gcf().set_facecolor('white')
plt.show()

### Figure SI4. The thickness of different ratios of SuperPThe thickness of different ratios of SuperP

In [ ]:
# 读取数据
path_thickness = Path(r"C:\Users\chengliu\OneDrive - UAB\ICMAB-Data\Zn-Mn\Cuatro\Result")
thickness = pd.read_excel(
    Path.joinpath(path_thickness, r"Thickness.xlsx"), sheet_name=0, index_col=None, header=0, engine="openpyxl"
)
# thickness

In [ ]:
import scipy


def linear_fit(x, a, b):
    return a * x + b


popt, pcov = scipy.optimize.curve_fit(
    linear_fit,
    xdata=thickness.iloc[:, 0],
    ydata=thickness.iloc[:, 1],
    method="trf",
    bounds=([-np.inf, -np.inf], [np.inf, np.inf]),
)

In [ ]:
# 计算预测值
y_pred = linear_fit(thickness.iloc[:, 0], *popt)

# 计算 SST 和 SSR
sst = np.sum((thickness.iloc[:, 1] - np.mean(thickness.iloc[:, 1])) ** 2)
ssr = np.sum((thickness.iloc[:, 1] - y_pred) ** 2)

# 计算 R^2
r_squared = 1 - (ssr / sst)
r_squared

In [ ]:
%matplotlib inline
plt.close("all")
fig = plt.figure(figsize=(3, 2))
gs = gridspec.GridSpec(1, 1, width_ratios=None, height_ratios=None, wspace=0, hspace=0, figure=fig)

# 图
subfig = fig.add_subfigure(gs[0, 0])
ax = subfig.add_axes((0, 0, 1.0, 1.0))
ax.set_box_aspect(0.8)

ax.scatter(thickness.iloc[:, 0], thickness.iloc[:, 1], ls="-", lw=1.0, c=colors[0], label=r"Data", marker="o")
ax.plot(thickness.iloc[:, 0], linear_fit(thickness.iloc[:, 0], *popt), ls="--", lw=1.0, c=colors[1], label=r"Fitting")

ax.set_xlabel(
    r"Mass Ratio (wt. %)",
    fontsize=11,
)
ax.set_xlim(-5, 105)
ax.xaxis.set_major_locator(ticker.MultipleLocator(base=20, offset=0))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(base=10, offset=0))

ax.set_ylabel(
    r"Thickness ($\mathrm{\mu m}$)",
    fontsize=11,
)
ax.set_ylim(30, 180)
ax.yaxis.set_major_locator(ticker.MultipleLocator(base=30, offset=0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(base=15, offset=0))

ax.tick_params(
    axis="both",
    which="both",
    labelsize=9,
    bottom=True,
    top=False,
    left=True,
    right=False,
    labelbottom=True,
    labelleft=True,
    labeltop=False,
    labelright=False,
)
ax.legend(loc="upper left", bbox_to_anchor=(0, 1), ncols=1, frameon=False, labelcolor="linecolor", fontsize=9)

ax.text(
    0.55, 0.2, f"y = {popt[0]:0.2f}*x + {popt[1]:0.2f}", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k"
)
ax.text(0.55, 0.12, f"R: {r_squared:0.3f}", ha="left", va="top", transform=ax.transAxes, fontsize=9, c="k")

plt.savefig(
    Path.joinpath(path_out, r"PaperCuatro_FigureSI_3_300_V1_0.tif"),
    pad_inches=0.0,
    bbox_inches="tight",
    dpi=300,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperCuatro_FigureSI_3_600_V1_0.tif"),
    pad_inches=0.0,
    bbox_inches="tight",
    dpi=600,
    transparent=False,
    pil_kwargs={"compression": "tiff_lzw"},
)
plt.savefig(
    Path.joinpath(path_out, r"PaperCuatro_FigureSI_3_900_V1_0.svg"),
    pad_inches=0.0,
    bbox_inches="tight",
    dpi=900,
    transparent=False,
)
plt.show()